# ASTR 457: Foundations of Data Science in Astronomy

**Fall 2026, University of Illinois Urbana-Champaign**

Prof. Gautham Narayan | TA: Abha Vishwakarma

Thu Aug 27, 2026 - Day 2: GitHub mechanics, probability, robust statistics, hypothesis testing

<img src="images/uiuc_logo.png" alt="University of Illinois Urbana-Champaign wordmark" width="220" style="display:block;margin:0 auto;">

## Today's plan

- **GitHub mechanics**: fork, your NetID folder, pull requests
- **Probability core**: axioms, distributions, descriptive statistics, the Central Limit Theorem **and when it doesn't hold**
- **Robust statistics**: why least-squares breaks, k-sigma clipping, and an actual live-fit mixture model
- **Hypothesis testing**: is that difference real or just noise, on real SDSS data

Pulled forward from the Sep 1/3 slot. Day 1 showed you are already comfortable with Python and AI tools, so we skip the planned Python crash course and start the statistics now.

One question runs through everything today: **how do you verify that your data, your fit, and your test are telling the truth?**

## Section 1: GitHub mechanics

## Why We Use GitHub

- everything for this course lives in one repo: <https://github.com/gnarayan/ast457_2026_Fall>
- syllabus, labs, help sheets, your own submitted work, all of it
- this is also just how real collaborative work happens, in industry and academia both
- if you don't know git yet, `help/git/` in the repo has cheat sheets - use them

## Step 1: Fork

<img src="images/github_mockup_fork.png" alt="GitHub Fork button, top right of the repo page" width="820" style="display:block;margin:0 auto;">

Click **Fork** (top right of the class repo page). GitHub creates a full
copy under *your own* account - you own this copy; the class repo is
untouched.

## Step 2: Clone it and make your folder

Everyone's folder lives under `submissions/` so the repo root stays clean
with ~20 of you. Two ways to create it - pick one.

**On your own machine**, with your fork's URL:

```
git clone https://github.com/<your-username>/ast457_2026_Fall.git
cd ast457_2026_Fall
mkdir -p submissions/netid
```

**Entirely in the browser.** GitHub's web UI has no empty-folder button.
Instead, create a new file and type a `/` in the filename - GitHub creates
the folder for you:

<img src="images/github_mockup_newfolder.png" alt="GitHub Create new file UI showing the path submissions/netid/lab01.ipynb" width="820" style="display:block;margin:0 auto;">

## Step 3: Copy in the lab, edit it, commit

```
cp labs/01/lab01.ipynb submissions/netid/lab01.ipynb
# ... do the lab, edit it in Jupyter ...
git add submissions/netid/lab01.ipynb
git commit -m "Lab 01: robust line fitting"
```

Same commands every week, only the lab number changes. Commit messages
don't need to be clever - they need to say what's in the commit.

## Step 4: Push to *your* fork

```
git push origin main
```

This pushes to your fork on GitHub, **not** to the class repo. Nothing
you push here is graded yet; it's just backed up and visible to you (and
GitHub) so far.

## Step 5: Open a pull request

<img src="images/github_mockup_pr.png" alt="GitHub banner after a push, offering Compare and pull request" width="820" style="display:block;margin:0 auto;">

After the push, GitHub shows this banner on your fork's page. Click
**Compare & pull request**, confirm the base repo is
`gnarayan/ast457_2026_Fall` on branch `main`, add a one-line description,
submit. **Only the PR gets graded - pushing by itself doesn't count.**

Abha and I review and **merge** your PR after grading. And yes, this is a
public repo: classmates can see your PR. That's fine by design - each of
you gets your own dataset, generated from your NetID (details when Lab 01
posts), so nobody else's numbers can help you.

## Step 6: Sync your fork before the next lab

<img src="images/github_mockup_syncfork.png" alt="GitHub Sync fork button showing the branch is behind the class repo" width="820" style="display:block;margin:0 auto;">

Your fork does **not** auto-update when the class repo changes. Before
starting each new lab: click **Sync fork → Update branch** on GitHub, or
locally:

```
git remote add upstream https://github.com/gnarayan/ast457_2026_Fall.git   # once, ever
git fetch upstream
git merge upstream/main
```

Because we merge your PR each week, syncing brings your fork level with the
class repo again - your next PR then contains only the new lab. Skip this
and next week's PR will show unrelated changes, or conflict outright.

## Expect to be messy at first

- most of you will accidentally push a file to the repo **root** instead of your folder the first time. That's normal, we'll fix it live
- a stale fork (didn't pull the latest class repo) is the single most common "why doesn't my lab folder look right" problem
- ask on Slack or in office hours before you're stuck for an hour - this is genuinely a five-minute fix once someone looks at it

## Section 2: Probability core

The repo is where your analysis will live. Now for what goes in it -
starting with the language every claim in this course is written in:
probability.

## Tonight's alert stream

Your survey flags 100 candidate transients tonight. Most are junk - satellite
glints, cosmic-ray hits, asteroids moving through the field. This is a real
problem every broker (ANTARES, ALeRCE, Fink...) has to solve, and it is the
example we use for the next few slides, through Bayes' rule.

For tonight's round numbers, call ~8 of the 100 real. (On real DECam
nights it's nearer ~1 in 100 - hold that thought.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n_alerts = 100
n_real = 8
labels = ['real transient', 'bogus\n(glint / CR / asteroid)']
counts = [n_real, n_alerts - n_real]

plt.bar(labels, counts, color=['C0', 'C3'])
plt.ylabel('alerts tonight')
plt.title("Tonight's sample space: 100 candidate alerts")
plt.show()

print(f"P(real)  = {n_real}/{n_alerts} = {n_real/n_alerts:.2f}")
print(f"P(bogus) = {n_alerts-n_real}/{n_alerts} = {(n_alerts-n_real)/n_alerts:.2f}")
print(f"P(real) + P(bogus) = {n_real/n_alerts + (n_alerts-n_real)/n_alerts:.2f}  "
      f"<- axiom 2 (P(sample space)=1) for a mutually-exclusive partition")


## What real and bogus actually look like

<img src="images/nair_real_vs_bogus.png" alt="Real vs bogus difference-imaging cutouts, viridis colormap" width="820" style="display:block;margin:0 auto;">

<small>Real vs. bogus difference-imaging cutouts. From the poster by Nair, Murphey, O'Brien &amp; Narayan (UIUC/CAPS, 2026), "Adapting Real-Bogus Classifier BRAAI for Transient Detection with DECam" - this is GN's own research group's actual work, adapting Duev et al. (2019)'s ZTF classifier to DECam.</small>

## What causes "bogus"

<img src="images/duev2019_fig4_real_bogus_examples.png" alt="Duev et al. 2019 Figure 4: real examples (supernova, variable star) vs bogus examples (badly-subtracted star, masked-bright-object artifact), science/reference/difference triplets" width="900" style="display:block;margin:0 auto;">

**A badly-subtracted star still leaves a real-looking residual. "Throw out
anything that looks weird" fails as a rule - for machines and for humans.**

<small>Figure 4 from Duev et al. (2019), MNRAS, arXiv:1907.11259, the
original BRAAI paper (the classifier the Nair poster adapted for DECam).
Left: real events - a supernova and a variable star. Right: bogus events - a
badly-subtracted star and a masked-bright-object artifact. These are the
"bad pixels, cosmic-ray hits, satellite trails" of Lab 01 and today's
robust-statistics section.</small>

## What the classifier sees

<img src="images/nair_three_channel_cutout.png" alt="Three-channel tmpl/img/diff cutout of a real DECam candidate" width="900" style="display:block;margin:0 auto;">

<small>Three-channel cutout of a real DECam candidate (cand: 6754). The source is visible in the center of both the template and image frames; the difference shows the residual of the two. The model's prediction: 1.00 (real). Nair, Murphey, O'Brien &amp; Narayan (UIUC/CAPS, 2026).</small>

## Axioms of probability

For a sample space $\Omega$ and event $A \subseteq \Omega$:

1. $P(A) \geq 0$
2. $P(\Omega) = 1$ (something in the sample space happens)
3. for mutually exclusive events, $P(A \cup B) = P(A) + P(B)$
   ($\cup$ reads "or": either event happens)

Everything else in probability (conditional probability, Bayes' rule, every distribution
we use this semester) is built on just these three statements.

## Two events at once: $P(A \cap B)$

$P(A \cap B)$ is the probability that A **and** B both happen, just as
$A \cup B$ meant "or" on the last slide.

- alert-stream version: A = "the transient is real", B = "the classifier
  flags it". $P(A \cap B)$ is the fraction of tonight's alerts that are
  real AND get flagged
- if A and B are **independent**, knowing one tells you nothing about the
  other, and the joint probability factorizes: $P(A \cap B) = P(A)\,P(B)$
- the whole point of a useful classifier is that "real" and "flagged" are
  NOT independent - the flag has to carry information. Conditional
  probability, next, is how we quantify that

## Conditional probability and Bayes' rule

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)} \quad\Longrightarrow\quad P(A \mid B) = \frac{P(B \mid A)\, P(A)}{P(B)}$$

- $P(A \mid B)$ reads "the probability of A, given that B happened"
- the arrow is two lines. The definition works both ways, because
  $A \cap B$ and $B \cap A$ are the same event:
  $P(A \cap B) = P(A \mid B)\,P(B)$ and $P(A \cap B) = P(B \mid A)\,P(A)$.
  Set the right-hand sides equal, divide by $P(B)$, and Bayes' rule falls out
- the denominator expands by the **law of total probability**,
  $P(B) = P(B \mid A)P(A) + P(B \mid \bar A)P(\bar A)$ - that's the
  expression the demo two slides from now computes as `p_flag`

## The same equation, wearing its working clothes

Swap the labels: $A \to H$ (a **Hypothesis**: "this transient is real") and
$B \to D$ (the **Data**: "the classifier flagged it"):

$$P(H \mid D) = \frac{P(D \mid H)\, P(H)}{P(D)}$$

- $P(H)$: the **prior**, what you believed about the hypothesis before this data
- $P(D \mid H)$: the **likelihood**, how probable the data is if the hypothesis holds
- $P(H \mid D)$: the **posterior**, what you believe after the data

Nothing changed but the letters. $A$ and $B$ were arbitrary labels, and so
are $H$ and $D$ - every textbook picks its own, and the structure
(posterior $\propto$ likelihood $\times$ prior) is the content. Today Bayes' rule gets two slides. From
the MCMC weeks onward it is the whole lecture - see the structure now, so it
is familiar then.

In [ ]:
# Your classifier flags a transient as "real".
# how good is that flag, really?
p_real = 0.08                # prior: ~8 in 100 (round demo numbers)
p_flag_given_real = 0.90     # true-positive rate, ~90% (demo)
p_flag_given_bogus = 0.20    # false-positive rate, ~20% (demo)

p_flag = p_flag_given_real * p_real + p_flag_given_bogus * (1 - p_real)
p_real_given_flag = p_flag_given_real * p_real / p_flag

print(f"P(real)              [prior]     = {p_real:.2f}")
print(f"P(real | flagged)    [posterior] = {p_real_given_flag:.2f}")

plt.bar(['prior\nP(real)', 'posterior\nP(real | flagged)'],
        [p_real, p_real_given_flag], color=['gray', 'C0'])
plt.ylabel('probability'); plt.ylim(0, 1)
plt.title("Bayes' rule: a decent classifier barely moves the needle\nwhen the base rate is low")
plt.show()

## What the real version of this achieved

<img src="images/nair_confusion_matrix.png" alt="Confusion matrix: 0 bogus predicted 0.99/0.01, 1 real predicted 0.00/1.00" width="620" style="display:block;margin:0 auto;">

The demo you just ran has a name: the **base-rate fallacy**. When real
events are rare, even an excellent classifier returns a modest posterior.
How much does a 99%-accurate classifier buy you? Next slide.

<small>Validation-set confusion matrix for the retrained BRAAI-on-DECam
classifier: 99% of true bogus correctly rejected, 100% of true real
candidates correctly recovered. Nair, Murphey, O'Brien &amp; Narayan
(UIUC/CAPS, 2026). Real numbers, real students, real telescope.</small>

## Rerun it with the real numbers

The demo used round numbers. The real classifier is far better - but the
Nair poster also reports the honest base rate: on DECam, only ~1 in 100
detections is real. Watch what happens.

In [ ]:
# same Bayes' rule, with the measured classifier numbers and the real base rate
p_real = 0.01                # ~1 in 100 DECam detections is real (Nair et al.)
p_flag_given_real = 1.00     # measured true-positive rate (validation set)
p_flag_given_bogus = 0.01    # measured false-positive rate

p_flag = p_flag_given_real * p_real + p_flag_given_bogus * (1 - p_real)
p_real_given_flag = p_flag_given_real * p_real / p_flag
print(f"P(real | flagged) = {p_real_given_flag:.2f}")
print("A ~99%-accurate classifier at a ~1% base rate: about HALF of the")
print("flags it raises are still bogus. Base rates are merciless -")
print("and this is why the humans have not been fully replaced.")

## Random variables

So far every event was yes/no: real or bogus, flagged or not. Measurements
are numbers - flux, magnitude, velocity. To put probabilities on numbers we
need one more idea.

- **discrete**: spectral type, number of detected photons in a bin
- **continuous**: magnitude, flux, parallax, radial velocity
- a random variable is described by its **distribution** - the function that tells you how
  likely each outcome (or range of outcomes) is

## PDF and CDF: the two ways to write down a distribution

For a continuous random variable $X$:

- the **probability density function (PDF)** $f(x)$: probabilities are
  *areas* under it, $P(a < X < b) = \int_a^b f(x)\,dx$. The density itself
  is never a probability - it can exceed 1, and $P(X = x)$ at any exact
  value is zero
- the **cumulative distribution function (CDF)**
  $F(x) = P(X \leq x) = \int_{-\infty}^{x} f(x')\,dx'$: starts at 0,
  climbs monotonically to 1
- **quantiles** run the CDF backwards: $x_p$ is the value where
  $F(x_p) = p$
- discrete variables use a probability *mass* function instead, where
  $P(k)$ really is a probability - the Poisson slide coming up is one

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

x = np.linspace(-4, 4, 400)
a, b = -1.0, 0.5
prob = stats.norm.cdf(b) - stats.norm.cdf(a)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, stats.norm.pdf(x), 'C0')
xs = np.linspace(a, b, 200)
axes[0].fill_between(xs, stats.norm.pdf(xs), alpha=0.4)
axes[0].set_title(f'PDF: shaded AREA = P({a} < X < {b}) = {prob:.2f}')
axes[0].set_xlabel('x'); axes[0].set_ylabel('density f(x)')

axes[1].plot(x, stats.norm.cdf(x), 'C1')
for v in (a, b):
    axes[1].axvline(v, color='gray', ls=':', lw=1)
axes[1].axhline(stats.norm.cdf(a), color='gray', ls=':', lw=1)
axes[1].axhline(stats.norm.cdf(b), color='gray', ls=':', lw=1)
axes[1].set_title(f'CDF: the same probability as a HEIGHT difference = {prob:.2f}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('F(x) = P(X \u2264 x)')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

x = np.linspace(-6, 6, 400)
fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), label='Gaussian (Normal)')
ax.plot(x, stats.t.pdf(x, df=2), label='Student-t, df=2 (fatter tails)')
ax.plot(x, stats.cauchy.pdf(x), label='Cauchy (df=1 Student-t)')
ax.set_xlabel('x'); ax.set_ylabel('probability density'); ax.legend()
ax.set_title('Same shape family, very different tails')
plt.show()

## Counting photons: the Poisson distribution

$$P(k \mid \mu) = \frac{\mu^k e^{-\mu}}{k!}, \qquad E(k) = \mathrm{Var}(k) = \mu$$

- the distribution of every photon count: CCD pixels, X-ray events, radio
  photon rates. If your data started as counts, it started as Poisson
- mean equals variance, so the fractional noise is $\sqrt{\mu}/\mu = 1/\sqrt{\mu}$:
  "shot noise"
- at large $\mu$ it converges to a Gaussian $\mathcal{N}(\mu, \sqrt{\mu})$, which is
  why "Gaussian noise" works for bright sources and breaks down for faint ones

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, mu in zip(axes, [2, 10, 50]):
    k = np.arange(0, int(mu + 5 * np.sqrt(mu) + 3))
    ax.bar(k, stats.poisson.pmf(k, mu), alpha=0.6, label=f'Poisson, mu={mu}')
    kk = np.linspace(k.min(), k.max(), 300)
    ax.plot(kk, stats.norm.pdf(kk, mu, np.sqrt(mu)), 'r-', label='Gaussian N(mu, sqrt(mu))')
    ax.set_xlabel('counts k'); ax.legend(fontsize=8)
plt.suptitle('Photon counting: Poisson converges to Gaussian as counts grow - faint sources are the hard case')
plt.tight_layout()
plt.show()

## Descriptive statistics

- sample mean: $\hat\mu = \bar X = \frac{1}{n}\sum_i X_i$
- sample variance: $\hat\sigma^2 = \frac{1}{n-1}\sum_i (X_i - \bar X)^2$
- **quantiles** $x_p$: where the CDF reaches $p$, i.e. $F(x_p) = p$;
  the median is $x_{0.5}$
- **interquartile range**: $\mathrm{IQR} = x_{0.75} - x_{0.25}$, a robust alternative to the
  standard deviation (you'll see why this matters a few slides from now, when the mean itself breaks)

In [ ]:
# where the mean, median, and IQR sit on skewed data
rng_d = np.random.default_rng(20260827)
flux = rng_d.lognormal(mean=0.0, sigma=0.9, size=20000)   # fluxes of a faint population
q25, q50, q75 = np.percentile(flux, [25, 50, 75])

plt.hist(flux, bins=120, density=True, alpha=0.6)
plt.axvspan(q25, q75, color='C2', alpha=0.15, label=f'IQR = [{q25:.2f}, {q75:.2f}]')
plt.axvline(q50, color='C2', lw=2, label=f'median = {q50:.2f}')
plt.axvline(flux.mean(), color='C3', lw=2, ls='--', label=f'mean = {flux.mean():.2f}')
plt.xlim(0, 8)
plt.xlabel('flux (arbitrary units)'); plt.ylabel('density'); plt.legend()
plt.title('Skewed data: the mean chases the tail; the median and IQR stay with the bulk')
plt.show()

## The Central Limit Theorem

If $X_1, \ldots, X_n$ are i.i.d. with $E(X_i) = \mu$ and $\mathrm{Var}(X_i) = \sigma^2$, then

$$\frac{\bar X - \mu}{\sigma/\sqrt{n}} \; \xrightarrow{\ n\to\infty\ } \; \mathcal{N}(0, 1)$$

**One reason the Gaussian is so important**: the *mean* of almost anything becomes
approximately Gaussian for large enough $n$ - even when the individual measurements
aren't Gaussian at all. Let's watch it happen.

In [ ]:
rng = np.random.default_rng(20260827)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n in zip(axes, [1, 5, 30]):
    # start from a very non-Gaussian population: a uniform distribution
    means = rng.uniform(0, 1, size=(20000, n)).mean(axis=1)
    ax.hist(means, bins=60, density=True, alpha=0.6)
    ax.set_title(f'mean of n={n} uniform draws')
plt.suptitle('CLT in action: sample means of a UNIFORM population converge to Gaussian')
plt.tight_layout()
plt.show()

## ...and when it doesn't hold

The CLT needs $E(X_i)$ and $\mathrm{Var}(X_i)$ to **exist and be finite**.

- most astronomical noise sources satisfy this, and the theorem is why "just average more
  exposures" usually works
- but some real distributions don't have a finite variance (or even a finite mean), and
  averaging more of them **does not help**
- the classic counterexample: the **Cauchy distribution** (a Student-t with 1 degree of
  freedom), heavier tails than any finite-variance distribution, and its mean is formally
  undefined
- the other assumption is **independence**: correlated measurements (time
  series, red noise) also break the naive $\sqrt{n}$ scaling - your *effective*
  $n$ is smaller than your row count. That failure mode gets weeks 8-11

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n in zip(axes, [1, 30, 1000]):
    means_gauss = rng.standard_normal(size=(5000, n)).mean(axis=1)
    means_cauchy = rng.standard_cauchy(size=(5000, n)).mean(axis=1)
    ax.hist(means_gauss, bins=np.linspace(-3, 3, 60), density=True, alpha=0.6, label='Gaussian pop.')
    ax.hist(np.clip(means_cauchy, -3, 3), bins=np.linspace(-3, 3, 60), density=True, alpha=0.6, label='Cauchy pop.')
    ax.set_title(f'n = {n}')
    if n == 1:
        ax.legend(fontsize=8)
plt.suptitle('The Gaussian-population mean tightens around 0 as n grows. The Cauchy-population mean does not.')
plt.tight_layout()
plt.show()

The lesson: averaging 1000 measurements of Gaussian noise buys you a factor of
$\sqrt{1000} \approx 32$ in precision, just as the CLT promises. Averaging 1000
measurements of Cauchy-distributed noise buys you **nothing** - the sample mean is just
as noisy at $n=1000$ as at $n=1$. Before you "just average more data," check that your
noise has a finite variance. This is the assumption robust statistics is built to protect you from
silently violating - starting on the next slide.

## When the mean fails, the median doesn't

Same Cauchy data. Take the **median** of each batch of draws instead of the
mean - and convergence comes right back. Quantile-based estimators (median,
IQR from the descriptive-statistics slide) survive tails that destroy
moment-based ones (mean, standard deviation). That is what "robust" means: an
estimator that a few wild points cannot drag away. The next section applies
the same principle to fitting a line - limit the influence of points that
violate your assumptions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n_draws in zip(axes, [1, 30, 1000]):
    draws = rng.standard_cauchy(size=(5000, n_draws))
    means_c   = np.clip(draws.mean(axis=1),   -3, 3)
    medians_c = np.clip(np.median(draws, axis=1), -3, 3)
    bins = np.linspace(-3, 3, 60)
    ax.hist(means_c,   bins=bins, density=True, alpha=0.55, color='C3', label='mean')
    ax.hist(medians_c, bins=bins, density=True, alpha=0.55, color='C2', label='median')
    ax.set_title(f'n = {n_draws} Cauchy draws')
    if n_draws == 1:
        ax.legend(fontsize=8)
plt.suptitle('The mean (red) never tightens. The median (green) does.')
plt.tight_layout()
plt.show()


## Diagnostic: the QQ-plot

Plot your data's quantiles against a reference distribution's quantiles.

- points falling on a straight line → your data is well-described by the
  reference distribution (scipy draws a fitted reference line, not literally 45°)
- points that peel away (usually in the tails) tell you *where* the assumption
  breaks

This is the fast visual check for "am I allowed to assume Gaussian here?" It's the
same question underneath every least-squares fit you'll do this semester.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
stats.probplot(rng.standard_normal(2000), dist='norm', plot=axes[0])
axes[0].set_title('Gaussian data vs. Gaussian reference\n(hugs the line)')
stats.probplot(rng.standard_cauchy(2000), dist='norm', plot=axes[1])
axes[1].set_title('Cauchy data vs. Gaussian reference\n(tails peel off hard)')
plt.tight_layout()
plt.show()

## Section 3: Robust statistics

## What least-squares actually assumes

Least-squares fitting makes three assumptions - our $y$ uncertainties are Gaussian, that the $x$ values are essentially exact, and every point's error is independent of every other point's.

Astronomical data routinely violates all three.

So this section is about what breaks, and about two standard fixes. K-sigma clipping is what Lab 01 Part 2 has you implement. The mixture model is what Part 3 grades you on understanding, and we're going to actually build that one rather than just describe it.

*Adapted from LSSTC-DSFP Session 7 Day0 "The Assumptions of Least Squares" (public, CC-licensed teaching material), reworked here as a live-taught demo rather than a fill-in-the-blank problem set.*


## A small dataset with a big problem

Twenty brightness measurements of a source, each with a stated uncertainty.

Some of them are contaminated. Bad pixels, cosmic-ray hits, a satellite trail, take your pick. We don't know which ones.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260827)

n = 20
x_true = np.linspace(0, 10, n)
y_true = 2.0 * x_true + 5.0
sigma = np.full(n, 1.5)
y = y_true + rng.normal(0, sigma)

# contaminate 4 points with a much larger, non-Gaussian excursion
bad = rng.choice(n, size=4, replace=False)
y[bad] += rng.choice([-1, 1], size=4) * rng.uniform(8, 15, size=4)

plt.errorbar(x_true, y, yerr=sigma, fmt='o', capsize=3)
plt.xlabel('x'); plt.ylabel('y')
plt.title('Raw data, which points do you trust?')
plt.show()

Just by eye, which points look wrong? Write down the ones you would throw out, before we run any code.


## Fit 1: least squares, everything included

This is what `np.polyfit`, or an AI assistant's first pass, hands us if we don't say anything about outliers.


## The reduced chi-squared statistic

$$\chi^2_{\rm red} = \frac{1}{n - k}\sum_i \frac{(y_i - \mathrm{model}(x_i))^2}{\sigma_i^2}$$

$n$ is the number of points and $k$ the number of fitted parameters. If the model is right and the $\sigma_i$ are honest, this lands near 1. Much bigger than 1 means the model is wrong, or the uncertainties are underestimated, or both.

How close to 1 counts as close? When everything is right, $\chi^2_{\rm red}$ still scatters by about $\sqrt{2/(n-k)}$, which is roughly $\pm 0.33$ for 20 points and 2 parameters. So 1.03 is unremarkable and 14 is a catastrophe.


In [ ]:
p_naive = np.polyfit(x_true, y, 1, w=1/sigma)
resid = y - np.polyval(p_naive, x_true)
chi2 = np.sum((resid / sigma) ** 2)
chi2_red = chi2 / (n - 2)
print(f"naive fit: slope={p_naive[0]:.2f}, intercept={p_naive[1]:.2f}, "
      f"chi2_red={chi2_red:.2f}  (true slope=2.00, intercept=5.00)")

xx = np.linspace(0, 10, 200)
plt.errorbar(x_true, y, yerr=sigma, fmt='o', capsize=3, label='data')
plt.plot(xx, np.polyval(p_naive, xx), 'r-', label='naive fit')
plt.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title(f'Naive fit, chi2_red = {chi2_red:.1f}')
plt.show()

## The same lesson, in the actual cited paper

<img src="images/hogg2010_fig2_naive_fit.png" alt="Hogg Bovy Lang 2010 Figure 2: naive weighted least-squares fit dragged off by outliers" width="760" style="display:block;margin:0 auto;">

<small>Figure 2 from Hogg, Bovy &amp; Lang (2010), arXiv:1008.4686, their own real 20-point example dataset (Table 1 in the paper). Same phenomenon: the naive weighted least-squares fit is visibly dragged off by a handful of outliers. This is the actual paper Lab 01 points you to.</small>


The $\chi^2_{\rm red}$ is way above 1, which is the same warning sign we got from Tuesday's demo.

The contaminated points are dragging both the slope and the intercept away from truth. They're dragging our uncertainty estimate too: a least-squares fit reports smaller error bars than it should, because it has no idea four of its inputs are lying to it.


## Fit 2: k-sigma clipping

Fit once, throw out anything more than $k\sigma$ away from the fit, then refit. Iterate until nothing new gets clipped.

This is what Lab 01 Part 2 asks you to implement.


## The k-sigma clipping criterion

Keep point $i$ only if

$$\left|\frac{y_i - \mathrm{model}(x_i)}{\sigma_i}\right| < k$$

We choose $k$ - the data does not derive it for us, and that choice is the method's weak point.


In [ ]:
def sigma_clip_fit(x, y, sigma, k=3.0, max_iter=10):
    mask = np.ones_like(x, dtype=bool)
    for _ in range(max_iter):
        p = np.polyfit(x[mask], y[mask], 1, w=1/sigma[mask])
        resid = (y - np.polyval(p, x)) / sigma
        new_mask = np.abs(resid) < k
        if np.array_equal(new_mask, mask):
            break
        mask = new_mask
    return p, mask

p_clip, mask = sigma_clip_fit(x_true, y, sigma, k=3.0)
resid = y[mask] - np.polyval(p_clip, x_true[mask])
chi2_red_clip = np.sum((resid / sigma[mask]) ** 2) / (mask.sum() - 2)
print(f"3-sigma clip: slope={p_clip[0]:.2f}, intercept={p_clip[1]:.2f}, "
      f"chi2_red={chi2_red_clip:.2f}, clipped {(~mask).sum()} of {n} points")
print(f"actually-contaminated indices: {sorted(bad)}; clipped indices: {sorted(np.where(~mask)[0])}")

plt.errorbar(x_true[mask], y[mask], yerr=sigma[mask], fmt='o', capsize=3, color='C0', label='kept')
plt.errorbar(x_true[~mask], y[~mask], yerr=sigma[~mask], fmt='x', capsize=3, color='r', label='clipped')
plt.plot(xx, np.polyval(p_naive, xx), 'r-', alpha=0.4, label='naive fit')
plt.plot(xx, np.polyval(p_clip, xx), 'C0-', label='clipped fit')
plt.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title(f'3-sigma clip, chi2_red = {chi2_red_clip:.2f}')
plt.show()

The threshold is $k$ measured in units of our stated $\sigma$. So there are two ways to get it wrong, and they turn out to be the same failure. We can pick a bad $k$, or we can feed it a wrong $\sigma$.

We'll look at both. First a wrong $\sigma$, over the next two slides, and then a live slider where you pick $k$ yourself.


## When our stated uncertainties are themselves wrong

If $\sigma$ is underestimated, clipping throws away good data. Points that are honestly a bit noisy start looking like outliers.

If $\sigma$ is overestimated, clipping keeps the bad data, because genuine contaminants no longer look discrepant enough to clip.

The trap is that from the fit alone both regimes can look clean, with a perfectly respectable $\chi^2_{\rm red}$, and we cannot tell which one we're in without independent knowledge of our uncertainties.

Watch the right panel on the next slide. It lands on $\chi^2_{\rm red} = 0.98$ with 3 of the 4 real contaminants still in the kept sample. <div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A good-looking $\chi^2_{\rm red}$ certifies nothing.**

</div>


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, factor, label in zip(axes, [0.3, 3.0], ['sigma underestimated (x0.3)', 'sigma overestimated (x3.0)']):
    sigma_wrong = sigma * factor
    p_w, mask_w = sigma_clip_fit(x_true, y, sigma_wrong, k=3.0)
    r_w = y[mask_w] - np.polyval(p_w, x_true[mask_w])
    cr_w = np.sum((r_w / sigma_wrong[mask_w]) ** 2) / (mask_w.sum() - 2)
    n_bad_kept = len(set(np.where(mask_w)[0]) & set(bad))

    ax.errorbar(x_true[mask_w], y[mask_w], yerr=sigma_wrong[mask_w], fmt='o', capsize=3, color='C0', label='kept')
    ax.errorbar(x_true[~mask_w], y[~mask_w], yerr=sigma_wrong[~mask_w], fmt='x', capsize=3, color='r', label='clipped')
    ax.plot(xx, np.polyval(p_w, xx), 'C0-', label='fit')
    ax.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
    ax.set_title(f'{label}\nclipped {(~mask_w).sum()}/{n}, chi2_red={cr_w:.2f}\n{n_bad_kept}/4 real contaminants still kept in')
    ax.set_xlabel('x')
axes[0].set_ylabel('y'); axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

Try it live. Instead of retyping `k=` and rerunning, drag the slider and watch which points get clipped, and what happens to the fit, as $k$ sweeps from aggressive to permissive.


In [ ]:
from ipywidgets import interact, FloatSlider

@interact(k=FloatSlider(value=3.0, min=1.0, max=6.0, step=0.25, description='k (sigma)'))
def explore_clipping(k):
    p, m = sigma_clip_fit(x_true, y, sigma, k=k)
    r = y[m] - np.polyval(p, x_true[m])
    cr = np.sum((r / sigma[m]) ** 2) / (m.sum() - 2) if m.sum() > 2 else np.nan

    plt.figure(figsize=(6, 4))
    plt.errorbar(x_true[m], y[m], yerr=sigma[m], fmt='o', capsize=3, color='C0', label='kept')
    plt.errorbar(x_true[~m], y[~m], yerr=sigma[~m], fmt='x', capsize=3, color='r', label='clipped')
    plt.plot(xx, np.polyval(p, xx), 'C0-', label='fit')
    plt.plot(xx, 2.0 * xx + 5.0, 'k--', alpha=0.5, label='truth')
    plt.xlabel('x'); plt.ylabel('y'); plt.legend(loc='upper left')
    plt.title(f'k={k:.2f}: clipped {(~m).sum()} of {n} points, chi2_red={cr:.2f}')
    plt.show()

## The better way: a mixture model (Hogg, Bovy & Lang 2010)

Same paper Lab 01 already points you to. Instead of making a hard in-or-out decision about each point, we model the data as coming from two populations at once.

There's a good population, Gaussian around the line, with the stated $\sigma_i$. And there's a bad population with its own Gaussian, mean $Y_b$ and variance $V_b$, entirely independent of the line.

Every point gets a probability of being bad, fit from the data. Nothing is manually thrown away, and the outlier fraction $P_b$ becomes a parameter we estimate instead of a knob we turn.

This is what Lab 01 Part 3 asks you to build. There your reported uncertainties are graded on calibration, meaning whether the truth lands inside your error bars as often as it should, so each point's outlier probability matters as much as the slope does. Let's fit one live.


## The mixture-model likelihood (Hogg, Bovy & Lang 2010)

$$\mathcal{L} = \prod_i \left[ (1-P_b)\, \mathcal{N}\!\big(y_i \mid mx_i+b,\ \sigma_i\big) \;+\; P_b\, \mathcal{N}\!\big(y_i \mid Y_b,\ \sqrt{V_b + \sigma_i^2}\big) \right]$$

The first term says point $i$ belongs to the good population, sitting on the line with its stated $\sigma_i$. The second says it belongs to the bad population, with its own mean and variance, unconnected to the line.

$P_b$, $Y_b$ and $V_b$ are all fit parameters, so the outlier fraction comes out of the data rather than being a choice we make.

The bad population's scale is $\sqrt{V_b + \sigma_i^2}$ because independent variances add. The measurement noise sits on top of the outlier population's own spread. And we fit $\ln V_b$ rather than $V_b$ so the optimizer can roam freely without ever proposing a negative variance.


In [ ]:
from scipy.optimize import minimize
from scipy.stats import norm

def neg_log_likelihood(theta, x, y, sigma):
    m, b, Pb, Yb, lnVb = theta
    Pb = np.clip(Pb, 1e-6, 1 - 1e-6)
    Vb = np.exp(lnVb)
    model = m * x + b
    good = (1 - Pb) * norm.pdf(y, loc=model, scale=sigma)
    bad_pop = Pb * norm.pdf(y, loc=Yb, scale=np.sqrt(Vb + sigma**2))
    return -np.sum(np.log(good + bad_pop + 1e-300))

# start from the sigma-clip fit, a sensible and defensible initial guess
theta0 = [p_clip[0], p_clip[1], 0.2, np.mean(y), np.log(50.0)]
result = minimize(neg_log_likelihood, theta0, args=(x_true, y, sigma), method='Nelder-Mead')
m_mix, b_mix, Pb_mix, Yb_mix, lnVb_mix = result.x
print(f"mixture fit: slope={m_mix:.2f}, intercept={b_mix:.2f}, "
      f"outlier fraction Pb={Pb_mix:.2f} (true fraction = {len(bad)/n:.2f})")

In [ ]:
# per-point posterior probability of belonging to the "bad" population
Vb_mix = np.exp(lnVb_mix)
model_mix = m_mix * x_true + b_mix
good_l = (1 - Pb_mix) * norm.pdf(y, loc=model_mix, scale=sigma)
bad_l = Pb_mix * norm.pdf(y, loc=Yb_mix, scale=np.sqrt(Vb_mix + sigma**2))
p_bad = bad_l / (good_l + bad_l)

plt.scatter(x_true, y, c=p_bad, cmap='coolwarm', s=80, edgecolor='k', vmin=0, vmax=1)
plt.colorbar(label='posterior P(outlier)')
plt.plot(xx, m_mix * xx + b_mix, 'g-', label='mixture fit')
plt.plot(xx, np.polyval(p_clip, xx), 'C0--', alpha=0.6, label='sigma-clip fit')
plt.plot(xx, 2.0 * xx + 5.0, 'k:', alpha=0.5, label='truth')
plt.xlabel('x'); plt.ylabel('y'); plt.legend()
plt.title('Mixture model: color = probability this point is an outlier')
plt.show()

print("point-by-point P(outlier):")
for i in np.argsort(-p_bad)[:6]:
    flag = 'REAL contaminant' if i in bad else 'clean'
    print(f"  point {i}: P(outlier)={p_bad[i]:.2f}  [{flag}]")

## ...and the actual paper's mixture-model result

<img src="images/hogg2010_fig4_mixture_fit.png" alt="Hogg Bovy Lang 2010 Figure 4: marginalized posterior for the mixture model and the MAP line" width="780" style="display:block;margin:0 auto;">

<small>Figure 4 from Hogg, Bovy &amp; Lang (2010). On the left, the marginalized posterior for (m, b) under the mixture (outlier) model. On the right, the MAP line in dark grey with draws from the posterior sampling in light grey. This is the professional-literature version of the plot you just watched fit live.</small>


#### What just happened, and why it's better

No point got permanently thrown away. Every point still carries some weight in the fit, in proportion to how likely it is to be good.

The outlier fraction $P_b$ came out of the fit rather than from a threshold we picked. Compare the printed probabilities against the truly contaminated indices above: the high-probability points should mostly be the real contaminants, and the fit degrades gracefully where the classification is uncertain instead of making a hard cut it can't explain.

One thing you should be complaining about. I quoted $P_b$ with no error bar on it, in this course of all places. Hogg's Figure 4 shows what getting one actually takes, which is the full posterior. That's what the MCMC weeks are for.


## Bridge to Lab 01

Sigma-clipping is the version we can write in five minutes and defend in one sentence. It's good enough when the answer isn't close to the threshold.

The mixture model is the version that doesn't make us pre-commit to a threshold at all. It reports the outlier fraction as a real parameter, and it degrades gracefully instead of making hard cuts.

You'll implement both in Lab 01. Now you've seen why the second one earns the extra twenty minutes it costs you.


## Section 4: Hypothesis testing

## Live exercise: your first package install

In your terminal, with `astr457` active:

```
conda install -c conda-forge astroml
```

Restart your kernel once it finishes.

This is the same workflow you'll use all semester whenever a lab needs a new package. Run `conda install`, confirm it landed in the right environment, then restart the kernel so the import actually picks it up.


## Meet the dataset: SDSS SEGUE Stellar Parameter Pipeline

327,260 Milky Way stars with SDSS spectra from Data Release 9. Each one has a pipeline-derived effective temperature, surface gravity, metallicity ([Fe/H]), alpha-element abundance ([alpha/Fe]), radial velocity, and proper motion.

The catalog was curated by Zeljko Ivezic, who is the first author of our textbook (Ivezic, Connolly, VanderPlas & Gray), and it's the dataset astroML's own worked examples are built around. We pull it through `astroML`, the companion package to the textbook, which you just installed.

These are real spectroscopic measurements of real stars. Every number we compute from here on is a measurement of an actual star in our own Galaxy.


In [ ]:
from astroML.datasets import fetch_sdss_sspp
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

data = fetch_sdss_sspp()
good = (data['FeHErr'] < 0.3) & (data['alphFeErr'] < 0.3) & (data['SNR'] > 20)
d = data[good]
print(f"{len(d)} stars after basic quality cuts")

plt.figure(figsize=(6, 5))
plt.hist2d(d['FeH'], d['alphFe'], bins=100, cmap='viridis')
plt.xlabel('[Fe/H]'); plt.ylabel('[alpha/Fe]')
plt.colorbar(label='N stars')
plt.title('The whole sample, before we split anything')
plt.show()

## Why this plot matters

We're looking at a direct measurement of Galactic archaeology: using the present-day chemistry of stars to reconstruct how the Milky Way disk formed.

The two streaks are two different stellar populations. One is metal-richer and lower-alpha, roughly the thin disk, younger, formed over a longer and more chemically enriched history. The other is metal-poorer and alpha-enhanced, roughly the thick disk, older, formed when the Galaxy had less time to enrich itself with iron before core-collapse supernovae seeded it with alpha elements.

[alpha/Fe] works as a rough clock because the alpha elements (O, Mg, Si, Ca, Ti) come promptly from core-collapse supernovae, while iron accumulates more slowly from Type Ia supernovae. So how alpha-enhanced a population is tells us something about how fast it formed its stars.

This is the split we'll use for today's hypothesis test. It's a real astrophysical population boundary drawn from the physics above, not an arbitrary demo cut.


## Why we have to be careful with this sample

This catalog is **not** a random sample of Milky Way stars. It's built from 10 explicit selection cuts, documented in the catalog's own header: a magnitude range, color cuts, proper-motion limits, a surface-gravity range, temperature-error cuts, and more.

The magnitude cut ($14 < r < 21$) is distance-dependent. It changes which stars we can see as a function of how far away they are, which is a classic Malmquist-type bias.

The proper-motion cut ($|\mu| < 200$ mas/yr) pre-selects on kinematics, so we cannot use this same sample to do an unbiased kinematic study without correcting for that cut first.

Any statistic we compute naively from this catalog, something like "what fraction of Milky Way stars are alpha-rich," is really answering a question about stars that pass these ten cuts.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**That gap between the sample and the population does not announce itself in the data.**

</div>


## The question

Split the stars by alpha-element abundance. An alpha-poor sample ([alpha/Fe] < 0.1, thin-disk-like) and an alpha-rich one ([alpha/Fe] > 0.3, thick-disk-like), the same two populations we just saw in the 2D histogram.

Do their metallicities come from different distributions, or could the difference in a histogram just be sampling noise?

Our **null hypothesis** $H_0$ is that the two [Fe/H] samples are drawn from the same underlying distribution. We ask the data to talk us out of it.

Full disclosure: that cut discards every star with $0.1 < [\alpha/\mathrm{Fe}] < 0.3$. Removing the overlap sharpens the contrast by construction. It's legitimate here, since we cut on $\alpha$ and test [Fe/H], but it is a choice, and per the previous slide it belongs in your write-up.


In [ ]:
alpha_poor = d[d['alphFe'] < 0.1]   # thin-disk-like
alpha_rich = d[d['alphFe'] > 0.3]   # thick-disk-like
print(f"alpha-poor: {len(alpha_poor)} stars, alpha-rich: {len(alpha_rich)} stars")

In [ ]:
for sample, color, name in [(alpha_poor, 'C0', 'alpha-poor'), (alpha_rich, 'C1', 'alpha-rich')]:
    feh = sample['FeH']
    q25, q50, q75 = np.percentile(feh, [25, 50, 75])
    plt.hist(feh, bins=60, density=True, alpha=0.5, color=color, label=name)
    plt.axvline(q50, color=color, lw=2)
    plt.axvspan(q25, q75, color=color, alpha=0.08)
    print(f"{name:11s}: median={q50:+.2f}, IQR=[{q25:+.2f}, {q75:+.2f}], mean={feh.mean():+.2f}")
plt.xlabel('[Fe/H]'); plt.ylabel('density'); plt.legend()
plt.title('Do these look like the same distribution? (line = median, band = IQR)')
plt.show()

By eye they look obviously different. But "obviously different" is exactly the kind of judgment call this course keeps asking us to back up quantitatively, the same theme as Tuesday's demo and Lab 01.

The printed medians and IQR bands are already most of the story, robust summaries on real data put to work two sections after we met them. The KS test turns that comparison into a single number.


## The Kolmogorov-Smirnov two-sample test

The KS test compares the two samples' **empirical CDFs** - the data's own stand-in for the CDF we met on Day 2. Sort the sample, step up by $1/n$ at each point.

Then it asks what the largest vertical gap between them is, and how often a gap that big would happen by chance if both samples really came from the same distribution.

Let's look at the ECDFs directly before we run the test. The test statistic is just a number on this plot.


In [ ]:
def ecdf(vals):
    s = np.sort(vals)
    return s, np.arange(1, len(s) + 1) / len(s)

x_poor, y_poor = ecdf(alpha_poor['FeH'])
x_rich, y_rich = ecdf(alpha_rich['FeH'])

plt.plot(x_poor, y_poor, label='alpha-poor ECDF')
plt.plot(x_rich, y_rich, label='alpha-rich ECDF')

# find and mark the largest gap, on a common grid
grid = np.linspace(min(x_poor.min(), x_rich.min()), max(x_poor.max(), x_rich.max()), 2000)
cdf_poor = np.searchsorted(x_poor, grid, side='right') / len(x_poor)
cdf_rich = np.searchsorted(x_rich, grid, side='right') / len(x_rich)
gap = np.abs(cdf_poor - cdf_rich)
i_max = np.argmax(gap)
plt.vlines(grid[i_max], cdf_rich[i_max], cdf_poor[i_max], color='k', linestyle='--',
           label=f'D = {gap[i_max]:.3f}')
plt.xlabel('[Fe/H]'); plt.ylabel('cumulative fraction'); plt.legend()
plt.title('The KS statistic is literally this gap')
plt.show()

## The KS statistic, formally

$$D = \max_x \big| F_1(x) - F_2(x) \big|$$

$F_1$ and $F_2$ are the two samples' empirical CDFs, so $D$ is just the largest vertical gap between the two curves we plotted. `scipy.stats.ks_2samp` computes $D$ and the p-value for us, and that plot is what it's doing underneath.


In [ ]:
ks_stat, p_value = stats.ks_2samp(alpha_poor['FeH'], alpha_rich['FeH'])
print(f"KS statistic = {ks_stat:.4f}  (matches the gap marked above)")
print(f"p-value = {p_value:.3e}")
# p prints as 0.000e+00 - the true value underflowed machine precision (see next slide).

## What does that p-value actually mean?

It is **not** the probability that the null hypothesis is true, and it is **not** the probability that the result is due to chance.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**If the two samples really were drawn from the same distribution, the p-value is the probability of seeing a gap $D$ at least this large, purely from sampling noise.**

</div>

Same discipline as the error-bar discussion on Tuesday. A p-value states what the data would look like under a specific assumption. It is not a verdict handed to us for free.

Here it's vanishingly small, so the null is not a reasonable description of what we see. The printout says $p = 0.000$, but a probability can't equal zero. The true value underflowed machine precision, so write $p < 10^{-300}$ and never $p = 0$.

Now watch what happens with much smaller samples of the exact same two real populations.


In [ ]:
rng = np.random.default_rng(20260827)
n_small = 8
sub_poor = rng.choice(alpha_poor['FeH'], n_small, replace=False)
sub_rich = rng.choice(alpha_rich['FeH'], n_small, replace=False)
ks_small_sample, p_small_sample = stats.ks_2samp(sub_poor, sub_rich)
print(f"with only {n_small} stars per sample: KS={ks_small_sample:.3f}, p={p_small_sample:.3f}")
print("Same real, genuinely-different populations. This p-value says 'not significant.'")

With only 8 stars per sample, the same test that was overwhelmingly significant on the full sample comes back saying it can't reject the null. That's a false negative, purely from bad luck in a small draw.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**A hypothesis test that fails to reject $H_0$ never means the two things are the same.**

</div>

It can just as easily mean our sample was too small to tell.

Sample size and test choice both belong in your write-up every time you report a p-value. It's exactly the kind of thing an AI assistant will not flag for you unprompted.


## The mirror trap: at n = 100,000, everything is significant

Our full-sample p-value underflowed to zero with 28k against 104k stars. But at that scale even a tiny, physically meaningless difference between two samples will reject the null too.

Statistical significance and scientific relevance come apart exactly when data are plentiful. That is the survey era. At LSST scale, "p < 0.05" is close to information-free.

So report the **effect size** alongside p. Here $D \approx 0.61$, meaning the two CDFs are separated by 61 percentage points at their widest, which is an enormous difference nobody would call subtle.

Small n hides real differences, as we just saw. Large n makes trivial ones scream.

<div style="font-size:1.35em; line-height:1.45; margin:0.6em 0;">

**The p-value on its own tells us neither.**

</div>


## One more trap: running the same test many times

Suppose that instead of one alpha cut we tried 20 different [Fe/H] and [alpha/Fe] binnings, looking for a significant difference somewhere.

At the standard $p<0.05$ threshold we'd expect about 1 of those 20 to look significant even if every population were drawn from the exact same distribution. That is what $p=0.05$ means.

This is the **multiple-comparisons problem** - it's what shows up when we let an AI assistant try a few different cuts and report what's significant, without telling it to correct for how many cuts it tried.

Rule of thumb for this course: if you ran more than one test, say so, and say how many. The simplest correction, Bonferroni, just demands $p < 0.05/20$ instead of $p < 0.05$.


In [ ]:
rng2 = np.random.default_rng(20260827)
n_trials, n_tests = 5000, 20
# simulate 5000 "nights": each night, run 20 tests where the null is TRUE
false_positives = (rng2.uniform(0, 1, size=(n_trials, n_tests)) < 0.05).sum(axis=1)

plt.hist(false_positives, bins=np.arange(-0.5, 8.5, 1), rwidth=0.8)
plt.xlabel('number of "significant" (p<0.05) results out of 20 true-null tests')
plt.ylabel('count over 5000 simulated repeats')
plt.title(f"mean = {false_positives.mean():.2f} false positives per 20 tests, purely by chance")
plt.show()

## Bridge: is a single population even Gaussian?

Before we fit anything with least-squares, today or in Lab 01, we are implicitly assuming Gaussian residuals. A QQ-plot is the fast visual check, the same tool from the probability core.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# left: the data itself, with the Gaussian we are implicitly assuming drawn over it
feh = alpha_poor['FeH']
mu, sd = feh.mean(), feh.std()
axes[0].hist(feh, bins=80, density=True, alpha=0.6, label='alpha-poor [Fe/H]')
xg = np.linspace(feh.min(), feh.max(), 400)
axes[0].plot(xg, stats.norm.pdf(xg, mu, sd), 'r-', lw=2,
             label=f'Gaussian, mean={mu:.2f}, sd={sd:.2f}')
axes[0].set_xlabel('[Fe/H]'); axes[0].set_ylabel('density'); axes[0].legend(fontsize=8)
axes[0].set_title('The data, with the Gaussian we assume')

# right: the same comparison, as a QQ-plot
stats.probplot(feh, dist='norm', plot=axes[1])
axes[1].set_title('The same comparison as a QQ-plot')
plt.tight_layout()
plt.show()

print(f"skew = {stats.skew(feh):+.2f}   (0 for a Gaussian)")


The verdict for [Fe/H] is that it is not Gaussian. The metal-poor tail peels away hard below the line, with a measured skew of about $-1.4$, so the median and IQR from the probability core earn their keep on this very sample.

In general, if the points hug the line then Gaussian is a fine working assumption. Where they peel off, usually in the tails, is where least-squares and "3-sigma means 99.7%" start lying to us.

That's the connective tissue between today's two segments. Hypothesis tests and QQ-plots are both verification tools for assumptions we would otherwise take on faith, the same checks Tuesday introduced, now on real SDSS data instead of a ten-point demo.


## Open questions a senior undergrad could actually work on

You now have every tool these projects need: the caveats, the test, and the verdicts.

* Correct the selection function. Forward-model the catalog's ten selection cuts through a Milky Way stellar population model and ask how the [Fe/H] and [alpha/Fe] distribution you'd infer changes once selection is accounted for. Well-posed and tractable.
* Cross-check against modern astrometry. This is DR9-era SEGUE data from around 2012, and Gaia now gives parallaxes and proper motions for the same stars. Do the thin and thick disk populations hold up once you add real distances and orbits, instead of inferring membership from chemistry alone?
* Chemistry against kinematics. This catalog already has radial velocities and proper motions in it, unused in today's demo. Do chemically-defined populations and kinematically-defined ones agree about which stars belong to which disk component? They don't always, and that disagreement is an active research question.
* Age-abundance relations. This catalog alone has no ages, but cross-matching against surveys that do, like asteroseismology or isochrone fitting, lets you test whether alpha-rich really always means old. That one is genuinely open right now.


## Wrapping up

The question from the first slide. How do we verify that our data, our fit, and our test are telling us the truth?

GitHub gives our analyses one home, in `submissions/<netid>/`, fork then PR. Bayes' rule turns classifier scores into what we actually want to know, and the CLT has fine print we need to check before leaning on it.

Robust statistics and hypothesis testing are the verification tools, the same theme as Tuesday's demo but now with the actual machinery behind it.

Lab 01 posts Thu Sep 3 and is due Wed Sep 9 at Noon, fork then PR. It asks you to implement both k-sigma clipping and the mixture model yourself, on your own per-NetID dataset.

Today's in-class exercises are optional take-home practice and are not collected. On Sep 1 and 3 we continue this deck from wherever today leaves off, on Zoom both days, link by email. Read `help/astro_conventions.md` before starting Lab 01.
